# Claude의 Agentic Search Benchmark 점수 재현

Claude의 [공개된 agentic-search 점수](https://www.anthropic.com/system-cards)(DeepSearchQA, BrowseComp)는 **공개 Messages API에서 재현할 수 있습니다**. 핵심은 harness 설정입니다. 짧은 대화에서는 중요하지 않은 몇 가지 API parameter가 agent가 수십만 token에 걸쳐 30회 이상의 tool call을 수행할 때는 핵심적인 역할을 합니다.

제3자 평가 결과가 더 낮은 흔한 이유는 harness 설정입니다. 이 cookbook은 각 parameter를 살펴보고 왜 중요한지 설명합니다.

> **비용 및 실행 시간**: 이 notebook의 실제 실행 셀은 **데모 질문 3개**를 실행합니다(약 5~10분, API 비용 몇 달러). 900개 질문의 전체 benchmark를 재현하려면 중간 수준의 concurrency에서 수백 달러와 몇 시간이 필요합니다. 이 notebook은 전체 실행을 다시 수행하지 않습니다.


## 이 cookbook을 마치면 다음을 할 수 있습니다.

- **구축**: programmatic tool calling, server-side compaction, task budget을 사용해 Claude의 공개된 agentic-search 점수를 재현하는 agentic search loop를 구축합니다.
- **이해**: 각 설정 선택이 긴 범위의 agentic 작업에 중요한 이유를 이해합니다.
- **조정**: dataset, 몇 줄의 설정, grader를 교체해 같은 harness를 BrowseComp 또는 자신의 deep-research benchmark에 맞춥니다.


## 사전 요구 사항

**필요한 지식**
- Messages API와 기본 tool use에 익숙함
- agentic loop 개념에 익숙함(model 호출 → tool use 처리 → 다시 호출)

**필요한 도구**
- Python 3.11+
- `anthropic >= 0.111.0`
- `web_search`, `web_fetch`, `code_execution` server tool에 접근할 수 있는 Anthropic API key([활성화 방법](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool))

**권장**
- 여기서 사용하는 패턴을 소개하는 [Programmatic Tool Calling cookbook](https://platform.claude.com/cookbook/tool-use-programmatic-tool-calling-ptc)
- 많이 활용하는 [Automatic Context Compaction cookbook](https://platform.claude.com/cookbook/tool-use-automatic-context-compaction)


## 설정


In [1]:
%%capture
%pip install -U "anthropic>=0.111.0" pandas python-dotenv

In [2]:
import time

import anthropic
import pandas as pd
from dotenv import load_dotenv
from utils.agentic_search import (
    SAMPLE_QUESTIONS,
    USER_PROMPT_TEMPLATE,
    extract_result_tag,
    grade_response,
    summarize_response,
)

load_dotenv()

MODEL = "claude-sonnet-5"
# 테스트 대상 model 간 점수를 비교할 수 있도록 grader model을 고정합니다.
GRADER_MODEL = "claude-opus-4-6"

# server-side 도구를 사용하면 단일 streaming 호출에서 첫 token이 도착하기 전에
# sandbox 내 도구 실행이 여러 분 동안 진행될 수 있습니다. read
# timeout을 60분으로 높입니다. streaming에서는 전체 응답이 아닌 event별
# 상한이므로 첫 event 이전의 간격에만 영향을 줍니다.
client = anthropic.Anthropic(
    max_retries=20,
    timeout=anthropic.Timeout(5.0, read=3600.0, write=600.0, pool=600.0),
)

## 작업

DeepSearchQA 형식의 데모 질문 세 개입니다. 각각 답할 수 있지만, 여러 차례 검색하고 읽고 교차 확인해야 합니다.


In [3]:
for q in SAMPLE_QUESTIONS:
    print(f"[{q['example_id']}] ({q['answer_type']})")
    print(f"  Q: {q['problem']}")
    print(f"  A: {q['answer']}\n")

[demo-0] (Single Answer)
  Q: Among the chief executives of the four largest US banks by total assets as of Q1 2024, which one had held their CEO position the longest?
  A: Jamie Dimon

[demo-1] (Set Answer)
  Q: Which three countries had the largest installed offshore wind capacity at the end of 2023?
  A: China, United Kingdom, Germany

[demo-2] (Set Answer)
  Q: List the films that won the Palme d'Or at the Cannes Film Festival in 2021, 2022, and 2023, and name the director of each.
  A: Titane (Julia Ducournau), Triangle of Sadness (Ruben Östlund), Anatomy of a Fall (Justine Triet)



각 질문을 짧은 프롬프트로 감싸서 Claude가 자유롭게 계획하고, grader가 깔끔하게 추출할 수 있도록 최종 답변을 `<result>` tag에 넣도록 요청합니다.


In [4]:
print(USER_PROMPT_TEMPLATE.format(question=SAMPLE_QUESTIONS[0]["problem"]))

I want you to answer the following question.

<question>Among the chief executives of the four largest US banks by total assets as of Q1 2024, which one had held their CEO position the longest?</question>

First plan out your response. This part can be as long as needed. You may need to run many searches, this is totally fine.

Then provide a short and concise answer in <result> tags. For questions expecting multiple answers, separate them with commas.


model은 계획하고, web을 검색하고, 페이지를 fetch하여 읽고, 계산을 실행하며, 확신이 생길 때까지 이를 계속한 다음 평가할 수 있는 깔끔한 최종 답변을 내보내야 합니다. 이 문장의 거의 모든 부분에서 단순한 harness는 눈에 띄지 않게 점수를 잃을 수 있습니다. loop를 하나씩 구축해 보겠습니다.

> 위 세 질문은 실제 실행 셀을 빠르고 저렴하게 유지하기 위해 이 cookbook용으로 작성했습니다(DeepSearchQA-900에서 가져온 것이 아님). 실제 benchmark는 아래 *Dataset*을 참조하세요.

### Dataset

| Benchmark | 규모 | 얻는 곳 |
|---|---|---|
| **BrowseComp** | 질문 1,266개 | [Wei et al., 2025](https://arxiv.org/html/2504.12516v1) (OpenAI) |
| **DeepSearchQA** | 질문 900개 | [Kaggle Benchmarks](https://www.kaggle.com/benchmarks/google/dsqa) ([논문](https://arxiv.org/abs/2601.20975)) |

harness는 각 행에 `problem`, `answer`, `answer_type` field가 있다고 가정합니다. 불러올 때 source schema를 이에 맞게 매핑하세요.


## 1. 도구: programmatic tool calling

가장 큰 단일 효율 개선 수단은 **[programmatic tool calling](https://platform.claude.com/docs/en/agents-and-tools/tool-use/programmatic-tool-calling)**(PTC)입니다. Claude가 `web_search`와 `web_fetch`를 직접 호출하고 각 결과를 API로 왕복시키는 대신 `code_execution` sandbox에서 Python을 작성하고 *그 코드 안에서* search/fetch를 호출합니다. 코드가 출력한 요약만 대화에 돌아옵니다.

deep research에서는 이것이 필수입니다. 질문 하나에서 50회 이상 fetch할 수 있습니다. PTC가 없으면 모든 페이지 본문이 context에 들어가고, PTC가 있으면 model이 sandbox 안에서 읽고 필요한 것만 꺼냅니다.

`code_execution`만 직접 호출 가능한 도구로 지정하고, `web_search`/`web_fetch`는 code execution*에서* 호출 가능하도록 표시하면서 결과를 응답에서 제외하여 활성화합니다.


In [5]:
TOOLS = [
    {
        "type": "code_execution_20260521",
        "name": "code_execution",
    },
    {
        "type": "web_search_20260318",
        "name": "web_search",
        "max_uses": 10_000,
        "allowed_callers": ["code_execution_20260521"],
        "response_inclusion": "excluded",
    },
    {
        "type": "web_fetch_20260318",
        "name": "web_fetch",
        "max_uses": 10_000,
        "max_content_tokens": 1_000_000,
        "allowed_callers": ["code_execution_20260521"],
        "response_inclusion": "excluded",
    },
]

BETAS = [
    "compact-2026-01-12",
    "task-budgets-2026-03-13",
]

짚어 둘 세부 사항 두 가지입니다.

- **`code_execution_20260521`**은 sandbox의 셀별 실제 시간 제한을 알려 주는 tool prompt를 사용하므로 Claude는 timeout되는 큰 loop 하나를 작성하는 대신 오래 걸리는 검색을 짧은 셀들로 나눕니다.
- **`response_inclusion: "excluded"`**는 `web_search`와 `web_fetch` 결과를 sandbox 안에 유지합니다. 응답의 나머지 모든 내용(`thinking`, `text`, `server_tool_use`, code-execution 결과, `compaction`)은 §3에서 `model_dump()`로 왕복합니다.


## 2. Thinking, effort, budget, compaction

요청 수준 parameter 네 개가 Claude에게 *얼마나 노력할지*와 *자신의 context를 어떻게 관리할지* 알려 줍니다.


In [6]:
THINKING = {"type": "adaptive"}

OUTPUT_CONFIG = {
    "effort": "max",
    "task_budget": {"type": "tokens", "total": 3_000_000},
}

# system-card 설정과 일치시킵니다. 다른 model의 점수를 재현하려면
# 해당 model card의 effort tier와 compaction trigger를 사용하세요.
COMPACTION_TRIGGER = 200_000

COMPACT_INSTRUCTIONS = (
    "Your summary MUST begin by restating the user's ORIGINAL QUESTION "
    "verbatim and in full, wrapped in <original_question> and "
    "</original_question> tags. Then summarize the research progress so "
    "far: key sources found, data extracted, partial answers, and what "
    "remains to be done. Do NOT call any tools. Your summary MUST also "
    "include this instruction verbatim: 'Provide your final answer "
    "wrapped in <result> and </result> tags.'"
)

CONTEXT_MANAGEMENT = {
    "edits": [
        {
            "type": "compact_20260112",
            "trigger": {"type": "input_tokens", "value": COMPACTION_TRIGGER},
            "instructions": COMPACT_INSTRUCTIONS,
        }
    ]
}

### `COMPACT_INSTRUCTIONS`가 매우 중요한 이유

대화가 trigger threshold에 도달하면 API는 이를 compact합니다. 별도 model 호출이 history를 요약하고 agent는 그 요약에서 계속합니다. 기본 요약 프롬프트는 설계상 작업에 종속되지 않습니다. *benchmark에서* 원래 질문과 답변 형식 지시가 반드시 살아남아야 할 두 가지라는 사실을 모릅니다. [사용자 지정 `instructions`](https://platform.claude.com/docs/en/build-with-claude/compaction#custom-summarization-instructions)로 이를 알려 줍니다.

이 지시가 없으면 compaction 이후 agent는 *찾아낸 것*의 요약을 갖지만 *요청받은 것*은 없으며, 긴 질문에서는 사용자에게 질문을 다시 말해 달라고 자주 요청하여 0점을 받습니다.

효과는 compaction이 발생하는 빈도에 따라 달라집니다. 낮은 trigger(질문 하나가 여러 번 compact될 수 있음)에서 가장 크고 여기서 사용하는 200K trigger에서는 더 작지만, 어느 경우든 harness에서 가장 저렴한 보험입니다.

> **참고**: 최신 Claude model card는 compaction *없이* DeepSearchQA를 보고합니다(model의 context window가 1M token이므로 1M-budget 작업이 한 window에 들어감). BrowseComp는 총 budget 3M에서 compaction*을 사용해* 보고합니다. 여기서는 compaction이 의미 있도록 3M-budget BrowseComp 설정을 사용합니다. DeepSearchQA에서는 여전히 대부분의 질문이 1M보다 훨씬 적은 범위에서 끝나므로 compaction을 trigger하지 않습니다.

나머지 세 가지:

| Parameter | 이유 |
|---|---|
| `thinking: adaptive` | 결과가 놀라울 때 Claude가 tool call 사이에 생각하고, 그렇지 않으면 생각을 건너뛰게 함 |
| `effort` | model card 수치는 model의 가장 높은 effort tier에서 보고됨 |
| `task_budget.total` | 모든 turn과 compaction에 걸친 누적 출력 budget을 Claude에게 알려 주어 일찍 포기하는 대신 속도를 조절할 수 있게 함 |


## 3. 왕복 loop

server-side 도구를 사용하면 각 API 호출은 내부적으로 여러 tool iteration을 실행한 뒤 checkpoint를 위해 `stop_reason="pause_turn"`으로 반환할 수 있습니다. client에서 할 일은 응답 content를 받아 assistant turn으로 추가하고 다시 호출하는 것뿐입니다. `stop_reason="end_turn"`까지 계속합니다.

helper 하나로 turn 사이의 request body를 깔끔하게 유지합니다.


In [7]:
def truncate_to_last_compaction(messages: list[dict]) -> list[dict]:
    """가장 최근 compaction block 앞의 기록을 제거.
    
    server는 compaction 시 그 기록을 이미 버렸으므로 다시 보내는 것은
    순수한 overhead이며 긴 실행에서는 32 MB request-body 한도를 넘을 수 있음.
    요청이 여전히 user turn으로 시작하도록 `messages[0]`(원래 user prompt)은
    항상 유지."""
    for mi in range(len(messages) - 1, 0, -1):
        content = messages[mi].get("content")
        if not isinstance(content, list):
            continue
        for bi in range(len(content) - 1, -1, -1):
            blk = content[bi]
            if isinstance(blk, dict) and blk.get("type") == "compaction":
                trimmed = {**messages[mi], "content": content[bi:]}
                return [messages[0], trimmed, *messages[mi + 1 :]]
    return messages

이제 loop 자체입니다. 길지만 모든 줄이 실제 benchmark 실행에 필요한 일을 하는 단일 함수입니다.


In [8]:
def sample(question: str, *, max_turns: int = 100) -> dict:
    """agentic search loop를 통해 질문 하나를 완료까지 실행."""
    # user message의 cache_control이 전체 요청의 prompt caching을 켜므로
    # server-side tool loop가 iteration 사이에 cache합니다.
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": USER_PROMPT_TEMPLATE.format(question=question),
                    "cache_control": {"type": "ephemeral"},
                }
            ],
        }
    ]
    request = {
        "model": MODEL,
        "max_tokens": 64_000,
        "tools": TOOLS,
        "betas": BETAS,
        "thinking": THINKING,
        "output_config": OUTPUT_CONFIG,
        "context_management": CONTEXT_MANAGEMENT,
        "messages": messages,
    }

    usage = {"input_tokens": 0, "output_tokens": 0, "cache_read_input_tokens": 0}
    transcript: list[str] = []
    tool_calls = 0
    start = time.time()

    for turn in range(max_turns):
        # 응답별이 아닌 event별 read-timeout이 적용되도록 streaming합니다.
        with client.beta.messages.stream(**request) as stream:
            response = stream.get_final_message()

        usage["input_tokens"] += response.usage.input_tokens
        usage["output_tokens"] += response.usage.output_tokens
        usage["cache_read_input_tokens"] += (
            getattr(response.usage, "cache_read_input_tokens", 0) or 0
        )
        tool_calls += sum(
            1 for b in response.content if getattr(b, "type", "") == "server_tool_use"
        )
        transcript.append(f"--- turn {turn} ---\n" + summarize_response(response))

        # turn 사이에 상태가 유지되도록 code-execution container를 유지합니다.
        if getattr(response, "container", None) is not None:
            request["container"] = response.container.id

        # 왕복: assistant content를 추가하고 compaction 이전 history를 제거합니다.
        messages.append(
            {
                "role": "assistant",
                "content": [b.model_dump(exclude_none=True) for b in response.content],
            }
        )
        messages[:] = truncate_to_last_compaction(messages)

        if response.stop_reason == "end_turn":
            final_text = "".join(
                b.text for b in response.content if getattr(b, "type", "") == "text"
            )
            return {
                "text": final_text,
                "result": extract_result_tag(final_text),
                "turns": turn + 1,
                "tool_calls": tool_calls,
                "elapsed_s": time.time() - start,
                "usage": usage,
                "transcript": transcript,
            }
        if response.stop_reason not in ("pause_turn", "max_tokens"):
            raise RuntimeError(f"unexpected stop_reason={response.stop_reason}")

    raise RuntimeError(f"did not finish within {max_turns} turns")

### Retry

SDK는 connection error, `408`, `409`, `429`, `≥500` 응답에 exponential backoff로 [자동 retry](https://platform.claude.com/docs/en/api/sdks/python#retries)를 수행하고, API가 rate-limit 응답에 보내는 `Retry-After` header를 따릅니다. 기본값은 retry 2회입니다. 전체 benchmark 실행에서는 실행 중 잠깐의 overload로 질문이 0점 처리되지 않도록 client에 `max_retries=20`을 설정합니다(위 *설정*).

> **눈에 띄지 않는 tool throttling 찾기**: server-tool rate limit은 model token limit과 별개입니다. 이를 소진해도 API error가 발생하지 않습니다. 응답 안에서 `too_many_requests`가 tool-result error로 나타나며 Claude는 보통 sandbox 안에서 retry합니다. 전체 benchmark 실행에서 exception 없이 예상보다 낮은 점수가 나오면 `result["transcript"]`에서 `rate_limit_error` 또는 `too_many_requests`를 출력하는 code-execution 셀을 확인하세요. client-side 해결 방법은 concurrency를 낮추는 것(또는 Anthropic support에 더 높은 tool rate limit 문의)입니다. API 호출이 200을 반환했으므로 client retry는 도움이 되지 않습니다.
>
> 마찬가지로 질문별 `sample()` 호출을 `try/except`로 감싸서 단일 `stop_reason="refusal"` 또는 복구 불가능한 오류가 전체 실행을 중단하는 대신 해당 질문을 0점 처리하도록 하세요.


## 4. 질문 하나를 end-to-end로 실행

첫 질문을 실행해 보겠습니다. 이 셀은 1~2분 걸릴 수 있습니다. 단일 `stream()` 호출 뒤에서 Claude가 검색 코드를 여러 번 작성하고 실행합니다.


In [9]:
result = sample(SAMPLE_QUESTIONS[0]["problem"])

print(f"tool_calls={result['tool_calls']}  elapsed={result['elapsed_s']:.0f}s")
print(
    f"tokens: in={result['usage']['input_tokens']:,}  out={result['usage']['output_tokens']:,}  "
    f"cache_read={result['usage']['cache_read_input_tokens']:,}"
)
print()
print("Final <result>:", result["result"])

tool_calls=4  elapsed=56s
tokens: in=130  out=3,842  cache_read=39,703

Final <result>: **Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).


transcript는 작업의 형태를 보여 줍니다. 각 turn은 sandbox 안의 tool call 묶음 이후의 `pause_turn` checkpoint입니다.


In [10]:
for line in result["transcript"]:
    print(line + "\n")

--- turn 0 ---
  [thinking]  963 chars
  [text]      I'll research this step by step: first identifying the four largest US banks by total assets as of Q
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  82 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  511 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  1365 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  700 chars
  [text]      Based on my research, I now have sufficient information to answer this question.  **Summary of findi
  [text]      "JPMorgan Chase & Co., the largest US bank by total assets, reported a sequential increase of $215.3
  [text]       were the other top banks, a ranking that held consistently across 2024.  Their respective CEOs and 
  [text]      Dimon became CEO on January 1, 2006 and one year later also became Chairman of the Board.
  [text]        2. **Bank of America – Brian Moynihan**: 
  [

## 5. 평가

benchmark 채점은 model-as-judge F1 grader를 사용합니다. grader는 질문, 정답, 추출된 `<result>` 텍스트를 보고 어떤 정답 항목이 발견됐는지, 정답 집합에 없는 추가 항목이 무엇인지 보고합니다. 이를 바탕으로 precision, recall, F1을 계산합니다.

올바르게 해야 할 두 가지:

- **전체 응답이 아닌 `<result>` tag를 평가하세요.** 전체 응답에는 계획에 관한 글과 검색 요약이 포함됩니다. 이를 grader에 제공하면 false positive와 false negative가 모두 증가합니다. `extract_result_tag()`는 최종 답변만 추출합니다.
- **grader model을 고정하세요.** `MODEL`과 관계없이 `GRADER_MODEL = "claude-opus-4-6"`을 사용합니다. 테스트 대상 model로 평가하면 model 간 점수를 비교할 수 없게 됩니다.

grader 프롬프트는 단순합니다.

~~~text
Your task is to evaluate whether a given response arrived at the correct answer.

Question: <question>{question}</question>
Correct answer (type: {answer_type}): <correct_answer>{answer}</correct_answer>
Response to evaluate: <response>{response}</response>

For each expected answer item, indicate whether it appears in the response.
Then list any answers in the response that are NOT in the correct-answer list.
Wording does not need to match exactly.

Reply in this exact XML format:
<evaluation>
  <explanation>one sentence</explanation>
  <correctness_details>
    <item answer="expected_item_1" correct="true|false"/>
  </correctness_details>
  <excessive_answers>
    <item>extra_item_if_any</item>
  </excessive_answers>
</evaluation>
~~~

Parsing과 precision/recall/F1 계산은 [`utils/agentic_search.py`](utils/agentic_search.py)에 있습니다. 여기서는 호출만 합니다.


In [11]:
grade = grade_response(client, GRADER_MODEL, SAMPLE_QUESTIONS[0], result["text"])
print(f"answer    : {grade['extracted_answer']}")
print(f"precision : {grade['precision']:.2f}")
print(f"recall    : {grade['recall']:.2f}")
print(f"F1        : {grade['f1']:.2f}")

answer    : **Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).
precision : 1.00
recall    : 1.00
F1        : 1.00


## 6. 데모 집합 실행

이제 세 질문 모두를 실행하고 비교 테이블에 사용량과 F1을 표시합니다. (900개 질문 전체 benchmark를 실행하려면 `SAMPLE_QUESTIONS`를 전체 dataset으로 바꾸고 concurrency를 추가하세요. *다음 단계* 참조.)


In [12]:
rows = []
for q in SAMPLE_QUESTIONS:
    r = sample(q["problem"])
    g = grade_response(client, GRADER_MODEL, q, r["text"])
    rows.append(
        {
            "id": q["example_id"],
            "tool_calls": r["tool_calls"],
            "in_tokens": r["usage"]["input_tokens"],
            "out_tokens": r["usage"]["output_tokens"],
            "f1": g["f1"],
            "answer": g["extracted_answer"][:60],
        }
    )

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nMean F1: {df['f1'].mean():.2f}")
# in_tokens는 cache되지 않은 청구 대상 입력입니다. caching을 켜면 실제
# context의 대부분은 cache read입니다(usage.cache_read_input_tokens에서 별도 추적).
print(f"Total tokens: {df['in_tokens'].sum():,} in / {df['out_tokens'].sum():,} out")

    id  tool_calls  in_tokens  out_tokens  f1                                                        answer
demo-0          11        312       10503 1.0  **Jamie Dimon**, CEO of **JPMorgan Chase** (the largest US b
demo-1           7        208        9768 1.0  The three countries with the largest installed offshore wind
demo-2           3        104        1592 1.0 - **2021**: *Titane* — directed by Julia Ducournau\n- **2022*

Mean F1: 1.00
Total tokens: 624 in / 21,863 out


## 7. 전체 benchmark로 확장

이 notebook의 설정을 전체 DeepSearchQA-900 및 BrowseComp-1266 집합에서 실행하면 공개된 model-card 점수가 재현됩니다. model별 공개 agentic-search 점수는 각 model의 [Claude model card](https://www.anthropic.com/system-cards)를 참조하세요.


## 8. BrowseComp에 맞게 조정

BrowseComp 질문은 집합형이 아닌 단일 답변형("1995년의 어떤 영화…")이므로 harness에서 바꾸는 것은 grader뿐입니다.


In [13]:
from utils.agentic_search import grade_browsecomp

# 평가: 한 글자로 판정(A=정답, B=오답, C=판정 유보).
# accuracy = "A" 판정의 비율.
bc_grade = grade_browsecomp(client, GRADER_MODEL, SAMPLE_QUESTIONS[0], result["text"])
print(bc_grade)

{'example_id': 'demo-0', 'extracted_answer': "**Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).", 'grader_letter': 'A', 'accuracy': 1.0}


도구, beta, `sample()` loop는 같습니다. 엄밀하게 재현하려면 model card에서 BrowseComp가 보고된 정확한 effort tier와 compaction trigger를 확인하세요. §2의 값은 좋은 기본값입니다.

핵심은 이것입니다. loop가 올바르면 benchmark를 교체하는 데 필요한 것은 dataset, 몇 줄의 설정, grader입니다.


## 다시 살펴보기

다음과 같은 agentic search loop를 구축했습니다.

- **code execution에서 programmatically** `web_search`와 `web_fetch`를 호출해 50회 이상의 page fetch가 context를 가득 채우지 않게 합니다(§1).
- Claude에게 **얼마나 노력할지**(`effort`, `task_budget`)와 **어떻게 생각할지**(`adaptive`) 알려 줍니다(§2).
- **compaction 프롬프트에 질문을 다시 명시**해 compaction 이후에도 유지합니다. 높은 compaction 빈도에서 가장 큰 단일 개선 수단입니다(§2).
- `pause_turn` 응답을 올바르게 왕복시키고 code container를 유지하며 compaction 이전 history를 잘라 냅니다(§3).
- model-as-judge F1 grader로 평가하고 공개된 DeepSearchQA 점수를 재현합니다(§5–7).


## 다음 단계

- **전체 benchmark로 확장**: `sample()`을 `concurrent.futures.ThreadPoolExecutor`로 감싸세요(client는 thread-safe). SDK의 내장 retry가 이미 429/529를 처리합니다. concurrency 50이면 DeepSearchQA-900은 대략 2시간에 끝납니다.
- **multi-agent 변형 시도**: [Async Multi-Agent Orchestration cookbook](https://platform.claude.com/cookbook/patterns-agents-async-multi-agent-orchestration)은 N-agent 팀이 더 낮은 latency로 이 single-agent loop의 정확도를 넘는 방법을 보여 줍니다.
- **자신의 benchmark로 이식**: contract는 `[{problem, answer, answer_type}]` 행, 몇 줄의 설정, grader 함수입니다. loop는 바뀌지 않습니다.
- **model card 읽기**: model별 [Claude model card](https://www.anthropic.com/system-cards)는 각 공개 benchmark 수치가 어떤 설정에서 나온 것인지 문서화합니다.
